# Starburst Amacrine Cell Direction Preference Sorting

Short workflow for sorting intracellular voltage-clamp responses from starburst amacrine cells.

## Data assumed

Input CSV:

```text
../data/starburst_direction_responses.csv
```

Rows:

```text
one cell per row
```

Columns:

```text
D0, D60, D120, D180, D240, D300
```

Optional:

```text
cell_id
```

## Goal

1. Determine preferred direction.
2. Compute direction selectivity index.
3. Cluster cells by six-direction response profile.
4. Sort and export results.

## Libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | file paths |
| `numpy` | DSI calculations |
| `pandas` | load/save tables |
| `matplotlib` | plots |
| `seaborn` | heatmap |
| `sklearn.preprocessing` | normalize response profiles |
| `sklearn.cluster` | unsupervised clustering |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Load data

This imports the already-cleaned response table.

In [ ]:
data_path = Path("../data/starburst_direction_responses.csv")

output_dir = Path("../outputs/starburst_direction_sorting")
fig_dir = output_dir / "figures"
table_dir = output_dir / "tables"

for folder in [output_dir, fig_dir, table_dir]:
    folder.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

if "cell_id" not in df.columns:
    df.insert(0, "cell_id", [f"cell_{i:04d}" for i in range(len(df))])

display(df.head())
print("Shape:", df.shape)

## Step 2 — Select direction columns

Edit `direction_cols` if your six direction columns have different names.

In [ ]:
direction_cols = ["D0", "D60", "D120", "D180", "D240", "D300"]
direction_degrees = np.array([0, 60, 120, 180, 240, 300])

missing = [c for c in direction_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing direction columns: {missing}")

for col in direction_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

responses = df[direction_cols].values.astype(float)

## Step 3 — Preferred direction and DSI

DSI:

```text
(preferred response - null/opposite response) / (preferred response + null/opposite response)
```

Interpretation:

```text
0 = weak direction preference
1 = strong direction preference
```

In [ ]:
def compute_direction_metrics(row_values):
    values = np.asarray(row_values, dtype=float)

    pref_idx = np.nanargmax(values)
    null_idx = (pref_idx + 3) % 6

    pref_response = values[pref_idx]
    null_response = values[null_idx]

    denom = pref_response + null_response

    if denom == 0 or np.isnan(denom):
        dsi = np.nan
    else:
        dsi = (pref_response - null_response) / denom

    return pd.Series({
        "preferred_direction_deg": direction_degrees[pref_idx],
        "preferred_response": pref_response,
        "null_direction_deg": direction_degrees[null_idx],
        "null_response": null_response,
        "DSI": dsi
    })

metrics = df[direction_cols].apply(compute_direction_metrics, axis=1)

df_metrics = pd.concat(
    [df[["cell_id"] + direction_cols], metrics],
    axis=1
)

display(df_metrics.head())

## Step 4 — Cluster cells by six-direction profile

This groups cells by response profile.

Responses are standardized so clustering is based more on profile shape than absolute current size.

In [ ]:
N_CLUSTERS = 4

X = df[direction_cols].values.astype(float)

col_means = np.nanmean(X, axis=0)
nan_rows, nan_cols = np.where(np.isnan(X))
X[nan_rows, nan_cols] = np.take(col_means, nan_cols)

X_scaled = StandardScaler().fit_transform(X)

clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS)
clusters = clusterer.fit_predict(X_scaled)

df_metrics["cluster"] = clusters

display(df_metrics[["cell_id", "preferred_direction_deg", "DSI", "cluster"]].head())

## Step 5 — Mean direction profile per cluster

This shows the average direction tuning for each discovered group.

In [ ]:
cluster_means = df_metrics.groupby("cluster")[direction_cols].mean()

plt.figure(figsize=(8, 5))

for cluster, row in cluster_means.iterrows():
    plt.plot(
        direction_degrees,
        row.values,
        marker="o",
        linewidth=2,
        label=f"Cluster {cluster}"
    )

plt.xlabel("Direction (degrees)")
plt.ylabel("Mean response")
plt.title("Mean direction tuning profile per cluster")
plt.xticks(direction_degrees)
plt.legend()
plt.tight_layout()

out_path = fig_dir / "cluster_mean_direction_profiles.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

## Step 6 — Sorted heatmap

Cells are sorted by:

```text
cluster → preferred direction → DSI
```

In [ ]:
ordered = df_metrics.sort_values(
    ["cluster", "preferred_direction_deg", "DSI"],
    ascending=[True, True, False]
).copy()

plt.figure(figsize=(7, max(4, len(ordered) * 0.15)))

sns.heatmap(
    ordered[direction_cols],
    cmap="viridis",
    yticklabels=ordered["cell_id"],
    cbar_kws={"label": "Response"}
)

plt.xlabel("Direction")
plt.ylabel("Cell")
plt.title("Sorted starburst direction responses")
plt.tight_layout()

out_path = fig_dir / "sorted_direction_response_heatmap.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

## Step 7 — Export sorted table

Main output:

```text
cell_id
six direction responses
preferred direction
DSI
cluster
```

In [ ]:
output_path = table_dir / "starburst_direction_preference_sorted.csv"

ordered.to_csv(output_path, index=False)

display(ordered.head(20))

print("Saved:", output_path)

## Interpretation

Use:

```text
preferred_direction_deg
```

to identify the strongest direction.

Use:

```text
DSI
```

to classify strong vs weak tuning.

Use:

```text
cluster
```

to group cells with similar six-direction tuning profiles.

This is a compact functional sorting workflow, not a heavy statistical model.